# 📗 임베딩 모델 선택: 리더보드에서 우리 평가셋까지

지금까지는 과정 기본 임베딩 모델(text-embedding-3-large, 768차원)을 정해진 대로 썼습니다. 하지만 리더보드 순위나 과정 기본값이 우리 문서와 질문에서도 가장 좋다는 보장은 없습니다.

**이 교안의 흐름**
- 리더보드로 후보를 좁힙니다.
- 국내 여행지 질문으로 후보 여섯(임베딩 후보 다섯과 BM25)의 정확도·질문 1건 평균 시간·비용을 같은 조건에서 측정합니다. 3-large는 과정 기본 768차원과 최대 3072차원을 따로 측정합니다.
- 차원을 줄였을 때 저장 공간과 정확도가 어떻게 바뀌는지 확인합니다.
- 임베딩 모델을 고른 결정을 ADR-001(결정 기록)로 남기고, 교안_02가 ADR-002~004를 이어 씁니다.

<img src="images/00_lesson01_goals.png" width="1050" alt="오늘 배울 것 세 가지: 1 리더보드 표에서 우리 문제에 맞는 점수 열만 돋보기로 골라 읽기, 2 같은 질문 카드를 다섯 후보 상자에 넣고 정확도·걸린 시간·비용을 함께 비교하기, 3 같은 모델의 벡터를 더 짧게(3072 -> 768차원) 받아 저장 공간과 정확도 비교하기">

**학습 목표** (그림의 번호와 같습니다)

1. 리더보드에서 우리 문제에 맞는 점수만 골라 읽고 후보를 정합니다.
2. 같은 평가셋으로 후보의 정확도·지연·비용을 측정해 비교하고, 고른 결과를 ADR로 남깁니다.
3. 차원을 줄였을 때 저장 공간과 정확도가 어떻게 바뀌는지 비교합니다.

## 1. RAG 아키텍처에서 고르는 것

무엇을 어떤 기준으로 고르는지 먼저 정해야 후보를 공정하게 비교할 수 있습니다.

### 1-1. 고를 대상 세 가지

RAG 아키텍처는 **임베딩 모델**(어느 모델로 벡터화할지), **벡터 저장소**(어디에 저장할지), **검색 전략**(질문마다 어떤 방법으로 찾을지) 세 가지 선택으로 정해집니다. 셋을 나눠 두어야 이 교안에서 임베딩 모델 하나만 바꿔 가며 비교할 수 있습니다.

<img src="images/01_three_choices.png" width="1050" alt="사내 도서관 책 찾기 서비스를 예로 든 세 층 건물. 위층부터 임베딩 모델, 벡터 저장소, 검색 전략을 층마다 하나씩 고른다">

여행지 안내 검색의 예:

1. **임베딩 모델**(교안_01): 관광지 388건과 질문을 어느 모델로 벡터화할지 정합니다.
2. **검색 전략**(교안_02): 질문 유형마다 키워드·의미 검색·하이브리드 가운데 무엇으로 찾을지 고르고, 필터와 관계·집계 질문을 어떻게 처리할지 정합니다.
    - "경복궁 근처에서 밤에 볼 곳"처럼 관계를 묻는 질문이나 "몇 곳이야?"처럼 세는 질문은 검색만으로 답하기 어려워 처리 방법을 따로 정합니다.
3. **벡터 저장소**(교안_02): 검색 전략이 보내는 지역 필터, 운영 정보 테이블 조인, 근처 관계를 처리하는 저장소를 고릅니다.

교안_02에서 검색 전략을 먼저 정하고, 그 전략이 보내는 질의로 벡터 저장소를 고른 뒤 세 가지를 하나의 아키텍처로 종합합니다. 앞 단원 기법이 RAG의 어느 단계에 들어가는지는 부록(RAG 단계별 기법 체계)에 정리했습니다.

### 1-2. 네 축으로 비교하기

후보는 **정확도, 지연, 비용, 저장 공간** 네 축으로 함께 비교합니다. 한 축만 보면 다른 축에서 감당할 수 없는 후보를 고를 수 있습니다.

네 축이 부딪히는 예:

- 후보 A는 두 문항을 더 맞히지만 질문마다 외부 API를 부릅니다.
- 후보 B는 API 요금이 없지만 문서 100만 건을 임베딩하려면 컴퓨터를 오래 돌려야 합니다.
- 3072차원 float32 벡터 1억 건은 벡터만 1억 × 3072 × 4 = 1,228,800,000,000바이트, 약 1.23TB입니다(1TB = 1조 바이트).

축의 무게는 서비스마다 다르고, 여러 축을 한 점수로 묶는 방법은 교안_02 6-4에서 다룹니다.

**정확도**: Recall@5, MRR, 맞힌 문항(정답 문서가 상위 5개 안에 든 문항 수)으로 측정합니다.

**비용**

색인은 문서를 한 번 임베딩해 벡터로 저장하는 일이고, 질의는 질문을 임베딩하는 일입니다.

| 항목 | 식 |
|---|---|
| API 색인 비용(달러) | 청크 수 × 청크당 평균 토큰 수 ÷ 1,000,000 × 100만 토큰당 단가 |
| API 질의 비용(달러) | 질문 수 × 질문당 평균 토큰 수 ÷ 1,000,000 × 100만 토큰당 단가 |
| 로컬 색인 시간(문서 임베딩) | (388건 문서 임베딩 시간 ÷ 388) × 청크 수 |
| 로컬 색인 비용(달러) | 색인 시간(시간) × 서버 시간당 요금 |

**저장 공간**
- 벡터 크기 = 청크(벡터) 수 × 차원 × 값 하나의 바이트(float32는 4바이트)
- 이 실습은 관광지(따라하기는 조문·매뉴얼 항목) 하나를 자르지 않고 통째로 임베딩해 문서 1개 = 청크 1개라, 문서 수와 청크 수가 같습니다.
- 실제 저장소에는 인덱스가 더해집니다. 교안_02 4-2에서 pgvector(PostgreSQL에 벡터를 저장·검색하는 확장) HNSW 인덱스 크기를 측정하고, 6절에서 그 배수로 100만 건 저장 공간을 계산합니다. 원문·메타데이터는 이 계산에 넣지 않습니다(검색 서버에 올리는 것만 셉니다).

### 1-3. 결정을 기록으로 남기기

세 선택은 한 번 정하면 되돌리기 비쌉니다.
- 임베딩 모델을 바꾸면 문서 전체를 다시 임베딩하고, 저장소를 바꾸면 데이터와 검색 코드를 옮겨야 합니다. 기본값으로 고른 모델이 우리 질문에 맞지 않으면 이 재임베딩 비용을 나중에 치릅니다.
- 이유를 적어 두지 않으면 두 가지 문제가 생깁니다.
    - 반년 뒤 "왜 이 모델을 골랐지?"라는 질문에 아무도 답하지 못해, 같은 비교를 처음부터 다시 합니다.
    - 결정이 기대던 조건(예: 질문을 회사 밖 API로 보내지 않는다)이 바뀌어도, 그 조건 때문에 내린 결정이라는 것을 아무도 몰라 다시 고를 때를 놓칩니다.

**ADR이란?**
- **ADR**(Architecture Decision Record, 아키텍처 결정 기록)은 결정 하나를 짧은 문서 한 장으로 남기는 기록입니다.
- ADR은 시스템 구조나 핵심 품질 특성(정확도·지연·비용·보안 등)에 영향을 주거나 되돌리기 어려운 결정에만 씁니다([Azure Well-Architected 가이드](https://learn.microsoft.com/en-us/azure/well-architected/architect-role/architecture-decision-record)). 기본값을 그대로 두는 작은 설정은 따로 ADR을 쓰지 않고 관련 ADR에 한 줄로 남깁니다(예: 검색 결과 수 k=5 같은 기본 설정).
- 오늘 두 교안은 부분이 끝날 때마다 결정을 ADR 한 장으로 남깁니다. 교안_01은 ADR-001(임베딩 모델), 교안_02는 ADR-002(검색 전략), ADR-003(벡터 저장소), ADR-004(최종 아키텍처)를 씁니다.
- 각 ADR의 제목에는 주제가 아니라 고른 기술을 적습니다(예: '벡터 저장소로 ○○ 사용').

**ADR 기본 형식**

실무에서는 Nygard가 제안한 다섯 칸 형식을 가장 널리 씁니다([원문](https://cognitect.com/blog/2011/11/15/documenting-architecture-decisions)).

| 칸 | 적는 것 |
|---|---|
| 제목 | 번호와 결정 내용(예: ADR-00N: 벡터 저장소로 ○○ 사용) |
| 상태 | 제안·채택·거절·대체됨 가운데 하나 |
| 맥락 | 결정이 필요한 상황과 지켜야 할 조건 |
| 결정 | 고른 것과 근거(고려한 후보와 각각 뺀 이유) |
| 결과 | 얻는 것과 감수하는 것, 다시 검토할 조건 |

- **다시 검토할 조건**: 어떤 가정이나 수치가 바뀌면 이 결정을 다시 내려야 하는지 적습니다. 예: 질문을 외부 API로 보내도 되면 API 모델도 다시 비교한다(6-4 노무 ADR-001).
- 결정 하나를 `docs/adr/0001-embedding-model.md`처럼 저장소의 파일 하나로 둡니다. 채택한 ADR은 고치지 않습니다. 결정이 바뀌면 새 ADR을 쓰고 옛 ADR의 상태를 대체됨으로 바꿉니다.

<img src="images/00_adr_cards.png" width="1050" alt="부분마다 결정을 ADR 한 장으로 남기기: 제목·상태·맥락·결정·결과 다섯 칸짜리 카드가 교안 01의 ADR-001 임베딩 모델과 교안 02의 ADR-002 검색 전략, ADR-003 벡터 저장소, ADR-004 최종 아키텍처로 이어진다">

## 2. 실습 준비

모든 후보를 같은 조건에서 측정하도록 모델, API 키, 자료를 먼저 준비합니다.

### 2-1. 설정

로컬 임베딩 모델 두 개를 미리 내려받습니다. 수업 전에 이 셀만 먼저 실행해 두면 뒤에서 기다리지 않습니다.

In [ ]:
from huggingface_hub import snapshot_download

# 로컬 임베딩 모델 두 개(합쳐 약 2.8GB) 미리 내려받기 (뒤 셀은 이 파일을 그대로 씀)
for model_name in ["nlpai-lab/KURE-v1", "intfloat/multilingual-e5-small"]:
    snapshot_download(model_name, ignore_patterns=["onnx/*", "openvino/*", "pytorch_model.bin"])  # 쓰지 않는 형식은 제외
    print(model_name, "준비 완료")

OpenAI 키는 앞 단원의 `.env`를 씁니다. 과정 기본 모델(`embedding_model`)과 비교 후보 3-small을 준비합니다.

**`check_embedding_ctx_length=False`란?**
- 최대 입력(8,192토큰)보다 긴 글을 나눠 평균 내는 기본 처리를 끕니다.
- 이 실습은 문서 하나를 청크 하나로 넣는데 가장 긴 문서도 1,545토큰이라 나눌 일이 없고, 넘는 글이 생기면 API 에러로 바로 드러납니다.

In [ ]:
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from langchain_openai import OpenAIEmbeddings

# data 폴더가 있는 단원 폴더 기준 경로
material_dir = Path(".")
data_dir = material_dir / "data"

# API 키 읽기: 단원 .env 먼저, 없는 공통 키는 저장소 최상위 .env에서 (이미 읽은 키는 덮어쓰지 않음)
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")

# 과정 기본 모델(3-large 768차원)과 비교 후보 3-small(기본 1536차원)
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large", dimensions=768,
                                   check_embedding_ctx_length=False)  # 긴 글을 나눠 평균 내지 않고 그대로 보냄
small_embedding_model = OpenAIEmbeddings(model="text-embedding-3-small", check_embedding_ctx_length=False)
print("임베딩 모델:", embedding_model.model, "| 차원:", embedding_model.dimensions, "| 비교 후보:", small_embedding_model.model)

### 2-2. 자료 살펴보기

모델을 평가할 질문을 고르려면 질문 유형과 정답 문서가 어떻게 담겼는지 먼저 봐야 합니다.

**`data/본실습/` 자료**
- `spots.jsonl`: 관광지 388건, 한 줄에 관광지 하나
- `questions.json`: 질문 75개, 질문마다 유형(`question_type`)과 정답 문서(`relevant_doc_ids`)

JSON 읽기는 앞 단원의 `read_json`을 씁니다.

In [ ]:
import json


def read_json(name):
    """data 폴더의 JSON 파일을 목록 또는 딕셔너리로 읽습니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


# 관광지와 질문 읽기 (spots.jsonl은 한 줄에 관광지 하나라 줄마다 읽음)
spots = [json.loads(line) for line in (data_dir / "본실습" / "spots.jsonl").read_text(encoding="utf-8").splitlines()]
questions = read_json("본실습/questions.json")

# 원문 ID와 임베딩 입력 (제목 + 줄바꿈 + 본문, 모든 후보가 같은 입력)
spot_ids = [spot["doc_id"] for spot in spots]
spot_texts = [spot["title"] + "\n" + spot["text"] for spot in spots]
print("관광지:", len(spots), "| 질문:", len(questions))

In [ ]:
import pandas as pd

display(pd.DataFrame(questions)["question_type"].value_counts().rename("질문 수").to_frame())  # 유형별 질문 수

**평가셋은 검색 한 번으로 답할 수 있는 묘사·이름 질문만 씁니다.** 그래야 결과 차이가 모델에서만 생깁니다(조건 질문은 교안_02 3절에서 측정하고, 관계·집계 질문은 교안_02 4-1에서 처리 방법을 다룹니다).

- 묘사: 본문과 다른 낱말로 설명합니다. 예: "희귀한 야생화와 곤충을 관찰하고 높은 흔들다리를 건너는 산속 명소는 어디야?"
- 이름: 그 본문에만 나오는 드문 고유명사로 찾습니다. 예: "‘추억의 거리’가 재현된 전통생활사 박물관은 어디야?"

평가 지표 함수는 검색 품질 평가 단원의 `recall_at_k`, `mrr`를 다시 정의합니다. 하이브리드 검색 단원은 검색 정확도를 Recall@3으로 측정했고, 오늘은 상위 5개를 LLM에 넘긴다고 보고 Recall@5로 측정합니다.

In [ ]:
# 검색 한 번으로 답할 수 있는 묘사·이름 질문만 평가셋으로
eval_questions = []
for question in questions:
    if question["question_type"] in ("묘사", "이름"):
        eval_questions.append(question)
eval_texts = [question["question"] for question in eval_questions]

print("평가셋:", len(eval_questions), "문항 | 1문항 =", round(100 / len(eval_questions), 1), "%p")  # 1문항이 Recall@5를 몇 %p 움직이는지

In [ ]:
# 검색 품질 평가 단원의 함수 (predicted는 검색이 낸 문서 id 목록(순위순), relevant는 정답 id 목록)
def recall_at_k(predicted, relevant, k):
    """전체 관련 문서 중 상위 k개가 찾아낸 비율(관련 수 / 전체 관련 수)."""
    hits = sum(1 for p in predicted[:k] if p in relevant)
    return hits / len(relevant)  # 분모는 그 질문의 정답 수 (정답 둘이면 0, 0.5, 1)

def mrr(predicted, relevant):
    """첫 번째 관련 문서 순위의 역수. 목록 안에 없으면 0."""
    # 검색 결과 전체(상위 10개)에서 첫 정답 찾기 (순위는 1부터)
    for i, p in enumerate(predicted, 1):
        if p in relevant:
            return 1 / i
    return 0.0

## 3. 리더보드로 후보 좁히기

공개된 임베딩 모델은 너무 많아 전부 우리 평가셋으로 측정할 수 없으므로, 리더보드로 정확도를 측정할 후보 서너 개를 먼저 고릅니다.

### 3-1. MTEB 읽고 후보 찾기

**MTEB**(Massive Text Embedding Benchmark)는 공개 데이터셋으로 임베딩 모델을 평가한 점수를 모은 리더보드입니다. 전체 순위는 우리 문제와 무관한 태스크까지 섞여 있으므로 아래 순서로 좁혀 읽습니다.

1. **태스크**: 검색(Retrieval) 점수만 봅니다. 분류·군집 점수는 검색이 아닌 다른 작업의 성능입니다.
2. **언어**: 한국어 벤치마크 MTEB(kor, v2)를 고릅니다(6유형 20개 태스크).
3. **데이터셋**: `PERFORMANCE PER TASK` 탭에서 데이터셋마다의 점수를 볼 수 있습니다. 우리 문서와 내용이 비슷한 검색 데이터셋의 점수를 봅니다(법령이면 LawIRKo, 공공 보건 안내면 PublicHealthQA).
4. **학습 데이터 표시**: `Only zero-shot` 같은 필터로 평가 데이터를 학습에 쓴 모델을 거릅니다. 개발자가 밝힌 정보로 판정하므로 보증은 아닙니다.

좁힌 모델 가운데 최대 입력, 라이선스, 크기, 가격이 맞는 서너 개를 남겨 우리 평가셋으로 정확도를 측정합니다. 리더보드 점수는 최종 근거가 아닙니다.

**직접 해 보기: 한국어 리더보드에서 후보 찾기**

1. 아래 주소의 MTEB(kor, v2) 요약 표를 엽니다.
    - 첫 열 `Rank`는 20개 태스크를 합친 전체 순위입니다.
    - 오른쪽 `Filters` 패널을 접으면 열이 더 보입니다.
2. `Retr.`(검색 태스크 9개의 nDCG@10 평균, 정답이 위에 있을수록 높음) 머리글을 눌러 정렬합니다.
    - `Rank`의 `#`는 정렬 뒤에도 전체 순위라, 검색 순위는 위에서부터 셉니다(`Retr.` 1위 KURE-v2가 `#38`).

<img src="images/hf_mteb_retr_sort.png" width="1050" alt="MTEB(kor, v2) 요약 표를 Retr. 열로 정렬한 화면. Retr. 열 머리글과 10번째 줄 nlpai-lab/KURE-v1 행에 빨간 테두리">

- 주소: <https://leaderboard.mteb.org/benchmark/MTEB%28kor%2C%20v2%29>
- 누를 곳: `Retr.` 열 머리글을 한 번 눌러 높은 점수 순으로 정렬

3. 상위 모델 다섯 개쯤의 이름을 눌러 아래 표를 채웁니다.
    - 차원은 `EMBEDDING DIM`, 최대 입력은 `MAX TOKENS`, 라이선스는 `License`입니다.
    - `Open weights`면 내 장비에서 돌릴 수 있고, `Proprietary`면 API로만 씁니다.

<img src="images/hf_mteb_model_page.png" width="1050" alt="리더보드의 nlpai-lab/KURE-v1 모델 페이지. Open weights 배지, License mit 줄, EMBEDDING DIM 1,024와 MAX TOKENS 8,192에 빨간 테두리">

- 주소: <https://leaderboard.mteb.org/models/nlpai-lab/KURE-v1>
- 누를 곳: 요약 표에서 모델 이름 `nlpai-lab/KURE-v1` 클릭

4. 과정에서 써 온 OpenAI 모델(`openai/text-embedding-3-small`, `openai/text-embedding-3-large`) 두 행은 미리 채워 두었습니다. 한국어 검색 태스크 9개 가운데 일부만 평가되어 `Retr.`가 비어 있습니다.

| Retr. 순위 | 모델 | Retr. | 파라미터 | 차원 | 최대 토큰 | 라이선스 | 가중치 공개 또는 API |
|---|---|---|---|---|---|---|---|
| | | | | | | | |
| 점수 없음 | openai/text-embedding-3-small | - | - | 1,536 | 8,191 | - | API |
| 점수 없음 | openai/text-embedding-3-large | - | - | 3,072 | 8,191 | - | API |

`Retr.`가 비어 있어도 같은 기준으로 비교할 수는 있습니다. `PERFORMANCE PER TASK` 탭에서 네 후보 모두 점수가 있는 검색 태스크만 고르면 데이터셋과 지표(nDCG@10)가 같습니다.

- `Retr.`: 한국어 검색 태스크 9개의 nDCG@10 평균입니다. 9개 중 하나라도 점수가 없으면 비어 있습니다.

| 데이터셋 | 자료 | 검색 과제 |
|---|---|---|
| `PublicHealthQA` | 미국 질병통제예방센터(CDC)·세계보건기구(WHO)의 공공 보건 FAQ, 한국어판 | 건강 질문으로 맞는 답변 문서 찾기 |
| `BelebeleRetrieval` | Meta가 122개 언어로 만든 독해 문제(Belebele), 위키 계열 기사 문단 | 독해 질문으로 답이 든 문단 찾기 |

| 모델 | `Retr.` | `PublicHealthQA` | `BelebeleRetrieval` |
|---|---|---|---|
| nlpai-lab/KURE-v1 | 75.90 | 81.92 | 92.65 |
| intfloat/multilingual-e5-small | 66.06 | 73.70 | 78.88 |
| openai/text-embedding-3-small | - | 70.65 | 67.97 |
| openai/text-embedding-3-large | - | 85.31 | 89.32 |

- 두 태스크에서 3-large와 KURE-v1이 앞서고, me5-small과 3-small이 뒤입니다. 두 태스크만의 결과라 우리 평가셋에서 다시 측정합니다.

- 주소: <https://leaderboard.mteb.org/benchmark/MTEB%28kor%2C%20v2%29>
- 누를 곳: `PERFORMANCE PER TASK` 탭 → 표 위 검색창에 모델 이름(예: `text-embedding-3`, `KURE-v1`)을 넣고 표를 오른쪽으로 밀어 `PublicHealthQA` 열을 읽습니다. `BelebeleRetrieval`은 같은 표의 왼쪽 두 번째 열입니다.

<img src="images/hf_mteb_task_openai.png" width="1050" alt="MTEB(kor, v2) PERFORMANCE PER TASK 탭. text-embedding-3을 넣은 검색창, PublicHealthQA 열 머리글, openai/text-embedding-3-large(85.31)와 openai/text-embedding-3-small(70.65) 행에 빨간 테두리">

- OpenAI가 발표한 MIRACL 평균(3-small 44.0, 3-large 54.9)은 다국어 검색 벤치마크라 데이터셋이 달라 이 비교에 넣지 않습니다.


<details><summary>예시 기록 보기</summary>

| Retr. 순위 | 모델 | Retr. | 파라미터 | 차원 | 최대 토큰 | 라이선스 | 가중치 공개 또는 API |
|---|---|---|---|---|---|---|---|
| 1 | nlpai-lab/KURE-v2 | 81.07 | 154M | 토큰마다 128 | 8,192 | apache-2.0 | 가중치 공개 |
| 2 | sionic-ai/comsat-embed-ko-8b-preview | 79.27 | 7.6B | 4,096 | 8,192 | cc-by-nc-4.0 | 가중치 공개 |
| 3 | Qwen/Qwen3-Embedding-8B | 78.26 | 7.6B | 4,096 | 32,768 | apache-2.0 | 가중치 공개 |
| 10 | nlpai-lab/KURE-v1 | 75.90 | 568M | 1,024 | 8,192 | mit | 가중치 공개 |
| 점수 없음 | openai/text-embedding-3-small | - | - | 1,536 | 8,191 | - | API |
| 점수 없음 | openai/text-embedding-3-large | - | - | 3,072 | 8,191 | - | API |

- KURE-v2(1위): 토큰마다 벡터를 만드는 late interaction 모델이라 문서당 벡터 하나인 저장소에 넣을 수 없습니다.
- comsat(2위): 비상업 라이선스(cc-by-nc-4.0)라 회사 서비스에 쓸 수 없습니다.
- Qwen3-Embedding-8B(3위): 파라미터 76억 개이고, 4,096차원이 pgvector HNSW `vector` 상한 2,000차원을 넘습니다.
- KURE-v1(10위): 3위와 2.36점 차이이고 노트북에서 돌아갑니다.
- OpenAI 모델: 한국어 점수가 없어 우리 평가셋으로 정확도를 직접 측정합니다. 최대 토큰은 리더보드의 8,191이 아니라 공식 문서의 8,192가 기준입니다.

</details>

**확인 퀴즈**: 리더보드 `Retr.` 1위 모델을 바로 우리 서비스에 쓰면 안 되는 이유 두 가지는 무엇일까요?

<details><summary>정답 보기</summary>

첫째, 공개 데이터셋 점수라 우리 문서와 질문에서도 순위가 같다는 보장이 없습니다. 둘째, 라이선스, 모델 크기, 차원 상한, 저장 방식처럼 점수에 드러나지 않는 조건이 맞지 않을 수 있습니다.

</details>

### 3-2. 후보 사양표

점수에 드러나지 않는 차원·최대 입력·가격·입력 규약을 한 표에 모아 두어야 뒤에서 잘림과 비용을 계산할 수 있습니다.

- API 모델(3-small, 3-large): 과정에서 써 온 기본값이고, 한국어 리더보드 점수가 없어 우리 평가셋으로 정확도를 직접 측정해야 합니다. 3-large는 과정 기본 768차원과 최대 3072차원 두 설정을 따로 측정합니다.
- KURE-v1: `Retr.` 상위권 가운데 노트북에서 돌릴 수 있는 한국어 검색 모델입니다(BGE-M3를 한국어 검색 데이터로 추가 학습).
- multilingual-e5-small: 점수는 낮은 편이지만 가장 가벼워, 작은 모델로 충분한지 봅니다.
- BM25: 하이브리드 검색 단원에서 배운 Kiwi 토큰 BM25를 오늘은 임베딩 모델이 키워드 검색보다 실제로 나은지 비교하는 기준선으로 씁니다. 둘을 합친 하이브리드의 정확도는 교안_02 3절에서 질문 유형별로 측정합니다.

| 모델 | 차원 | 최대 입력 | 가격 | 라이선스 | 입력 규약 |
|---|---|---|---|---|---|
| [text-embedding-3-small](https://developers.openai.com/api/docs/guides/embeddings) | 1536 (dimensions로 줄일 수 있음) | 8192 토큰 | 100만 토큰당 $0.02 | OpenAI API로 사용 | 질문·문서 그대로 |
| [text-embedding-3-large](https://developers.openai.com/api/docs/models/text-embedding-3-large) (과정 기본) | 768 (`dimensions=768`) | 8192 토큰 | 100만 토큰당 $0.13 | OpenAI API로 사용 | 질문·문서 그대로 |
| [text-embedding-3-large](https://developers.openai.com/api/docs/models/text-embedding-3-large) (최대 차원) | 3072 | 8192 토큰 | 100만 토큰당 $0.13 | OpenAI API로 사용 | 질문·문서 그대로 |
| [intfloat/multilingual-e5-small](https://huggingface.co/intfloat/multilingual-e5-small) | 384 | 512 토큰 (넘으면 뒤가 잘림) | 내 컴퓨터에서 실행, 가중치 파일 약 0.47GB | MIT | 질문 앞에 `query: `, 문서 앞에 `passage: ` |
| [nlpai-lab/KURE-v1](https://huggingface.co/nlpai-lab/KURE-v1) | 1024 | 8192 토큰 | 내 컴퓨터에서 실행, 가중치 파일 약 2.3GB | MIT | 질문·문서 그대로 |
| [BAAI/bge-m3](https://huggingface.co/BAAI/bge-m3) (KURE의 바탕 모델, 이번 비교에서 제외) | 1024 | 8192 토큰 | 내 컴퓨터에서 실행 | MIT | 질문·문서 그대로 |
| BM25 (기준선) | 없음 | 제한 없음 | 내 컴퓨터에서 실행 | - | 질문·문서를 낱말로 나눔 |

**최대 입력 토큰이란?**
- 입력 하나에서 읽는 토큰 수의 상한입니다.
- 토큰은 모델마다 자기 토크나이저로 세므로 같은 글도 모델마다 토큰 수가 다릅니다.
- 청킹 전략 단원에서 정한 청크 길이도 이 상한 안이어야 잘리지 않습니다. 오늘은 같은 문서가 상한이 작은 모델에서 잘리는지를 잘림 비율로 봅니다.
- 잘림은 더 잘게 청킹해 풀 수도 있지만, 벡터 수가 늘어 저장 공간(5절)과 색인 비용이 커집니다.

**모델별 상한**
- OpenAI 모델은 상한을 넘으면 API가 에러를 내고, multilingual-e5-small은 512토큰 뒤를 에러 없이 잘라 냅니다(비교표의 `truncated_ratio`).
- 상한과 견줄 길이는 임베딩에 넣는 단위인 청크의 길이입니다. 이 실습은 문서 하나를 청크 하나로 넣으므로 문서 길이로 봅니다(OpenAI 기준 관광지 평균 474·최대 1,545토큰, 노무 평균 302·최대 1,236토큰).

**KURE-v1 모델 카드의 자체 평가**(한국어 검색 데이터셋 8개 평균)
- 상위 5개 Recall은 KURE-v1 0.739, text-embedding-3-large 0.664입니다.
- 열 머리글은 `Average Recall_top1`이지만 Top-k 5 표의 값입니다.

<img src="images/hf_kure_card_topk5.png" width="520" alt="Hugging Face KURE-v1 모델 카드의 Top-k 5 표. Average Recall_top1 머리글, nlpai-lab/KURE-v1 행(0.73851), openai/text-embedding-3-large 행(0.66365)에 빨간 테두리">

- 주소: <https://huggingface.co/nlpai-lab/KURE-v1>
- 누를 곳: 카드 아래쪽 `Results`의 `Top-k 5` 표

multilingual-e5는 질문 앞에 `query: `, 문서 앞에 `passage: `를 붙여야 합니다.

<img src="images/hf_e5_card_prefix.png" width="640" alt="Hugging Face multilingual-e5-small 모델 카드 FAQ 1번. query: 와 passage: 접두어를 붙여야 하는지 묻는 질문에 빨간 테두리, 답은 Yes">

- 주소: <https://huggingface.co/intfloat/multilingual-e5-small>
- 누를 곳: 카드 아래쪽 `FAQ` 1번

## 4. 우리 질문으로 후보 평가하기

리더보드 점수는 공개 데이터셋 기준이라, 같은 평가셋으로 정확도를 직접 측정해야 우리 질문에서 어느 후보가 나은지 알 수 있습니다.

**여행지 안내 챗봇의 가정**
- 관광지 개요를 끝까지 읽고 답해야 합니다.
- 이용자 질문을 회사 밖 API로 보내지 않습니다.
- '부산의 박물관'처럼 지역으로 거르는 질문, 휴무일 같은 운영 정보를 함께 보는 질문, '근처'를 묻는 질문이 많습니다.
- 회사는 이미 PostgreSQL을 운영합니다.

임베딩 모델은 앞의 두 가정으로 고릅니다(뒤의 두 가정은 교안_02 4절에서 저장소를 고를 때 씁니다). BM25는 고를 대상이 아니라 기준선입니다.

**선정 기준**

1. 정확도를 가장 중요하게 봅니다.
    - 맞힌 문항이 가장 많은 후보와 2문항 이내 차이는 같은 수준으로 봅니다(1문항 = Recall@5 3.3%p라 1~2문항은 평가셋이 바뀌면 뒤집힙니다).
2. 개요를 끝까지 읽어야 하므로 잘림 비율이 1%를 넘는 모델은 뺍니다.
3. 질문을 회사 밖으로 보내지 않으므로 같은 수준이면 로컬 모델을, 그다음 질문 1건 평균 시간이 짧은 모델을 고릅니다.

가정이 다르면 순서도 달라집니다. 예를 들어 장비를 둘 수 없고 질문을 외부로 보내도 되는 서비스라면 3번에서 API 모델을 고를 수 있습니다.

<img src="images/04_lesson01_selection.png" width="1050" alt="임베딩 모델 선정 순서: 정확도로 거르기(2문항 이내는 같은 수준) → 잘림 1% 넘는 후보 빼기 → 로컬·짧은 시간 순으로 고르기">

### 4-1. 같은 조건으로 비교하는 함수

모델만 비교하려고 나머지 조건을 같게 둡니다. 저장소의 근사 검색(HNSW)은 가끔 상위 문서를 놓치므로, NumPy로 모든 문서와 비교하는 **정확 검색**을 씁니다.

- `normalize`: 벡터 길이를 1로 맞춥니다. 그러면 내적이 코사인 유사도입니다.
- `search_top_ids`: 질문마다 상위 `top_k`개 원문 ID를 순위대로 돌려줍니다.
- `evaluate_model`: 검색 결과를 정답과 비교해 비교표 한 행을 만듭니다. BM25도 평가하려고 벡터 대신 검색 결과를 받습니다.

RAG에서는 LLM이 상위 k개만 읽으므로 Recall@k를 먼저 보고 MRR은 보조로 봅니다.

In [ ]:
import numpy as np


def normalize(vectors):
    """행마다 길이를 1로 맞춘 벡터 배열을 반환합니다."""
    vectors = np.asarray(vectors, dtype=np.float32)  # 임베딩 API가 돌려준 리스트를 float32 배열로
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)  # 길이가 1이면 내적이 곧 코사인 유사도


def search_top_ids(question_vectors, doc_vectors, doc_ids, top_k=10):
    """질문마다 내적이 큰 문서 top_k개의 원문 ID를 순위대로 담은 목록을 반환합니다."""
    scores = question_vectors @ doc_vectors.T  # (질문 수, 문서 수) 점수표, 모든 문서와 비교하는 정확 검색

    # 질문마다 점수가 큰 문서 top_k개를 원문 ID로
    top_ids = []
    for question_scores in scores:
        order = np.argsort(-question_scores, kind="stable")[:top_k]  # 동점이면 문서 순서 유지(stable)
        top_ids.append([doc_ids[position] for position in order])
    return top_ids


def evaluate_model(name, top_ids):
    """평가셋 질문별 검색 결과를 정답과 비교해 Recall@5·MRR·맞힌 문항을 담은 한 행(dict)을 반환합니다."""
    # 질문마다 Recall@5·MRR을 모으고 맞힌 문항 세기 (top_ids는 eval_questions와 같은 순서)
    recalls = []
    reciprocal_ranks = []
    found = 0
    type_recalls = {"묘사": [], "이름": []}
    for question, retrieved_ids in zip(eval_questions, top_ids, strict=True):
        recall = recall_at_k(retrieved_ids, question["relevant_doc_ids"], 5)
        recalls.append(recall)
        reciprocal_ranks.append(mrr(retrieved_ids, question["relevant_doc_ids"]))  # 상위 10개 안 첫 정답의 순위로
        type_recalls[question["question_type"]].append(recall)
        if recall > 0:  # 정답 문서가 하나라도 상위 5개 안이면 맞힌 문항
            found += 1

    # 평균을 소수점 셋째 자리로 반올림해 비교표 한 행으로
    return {
        "model": name,
        "mean_recall_at_5": round(float(np.mean(recalls)), 3),
        "mrr": round(float(np.mean(reciprocal_ranks)), 3),
        "맞힌 문항": found,
        "묘사 R@5": round(float(np.mean(type_recalls["묘사"])), 3),
        "이름 R@5": round(float(np.mean(type_recalls["이름"])), 3),
    }

### 4-2. 후보별로 임베딩하고 시간 측정하기

정확도만으로는 지연과 비용을 비교할 수 없으므로, 후보마다 문서 388건을 임베딩하며 걸린 시간을 측정합니다. 질문은 하나씩 처리하면서 질문 1건 평균 시간을 함께 측정합니다.

**기준선: BM25.** 임베딩 없이 낱말 겹침으로 찾는 기준선입니다. 토큰화는 하이브리드 검색 단원의 `kiwi_tokenize`입니다.

In [ ]:
import time
from kiwipiepy import Kiwi
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever

kiwi = Kiwi()  # BM25 토큰화에 쓸 한국어 형태소 분석기

# 후보 이름을 키로 결과 모으기 (뒤의 모델 비교표가 읽음)
top_ids_by_model = {}   # 질문별 상위 10개 원문 ID
doc_seconds = {}        # 문서 388건 색인(문서 임베딩, BM25는 색인 만들기)에 걸린 초
query_ms = {}           # 질문 1건 평균 시간(ms)


def kiwi_tokenize(text):
    """문서와 질문에서 명사·외국어·숫자를 같은 기준으로 추출합니다."""
    return [token.form.lower() for token in kiwi.tokenize(text.replace("･", "·"))  # 반각 가운뎃점(･)은 ·로 맞춤
            if token.tag.startswith("N") or token.tag in {"SL", "SN"}]  # N*: 명사, SL: 외국어, SN: 숫자


spot_documents = [Document(id=doc_id, page_content=text, metadata={"source_id": doc_id}) for doc_id, text in zip(spot_ids, spot_texts)]  # 원문 ID를 source_id에 담음

# BM25 색인과 걸린 시간 (k=10: MRR을 10위까지 보므로 상위 10개를 돌려받음)
start = time.perf_counter()
bm25 = BM25Retriever.from_documents(spot_documents, preprocess_func=kiwi_tokenize, k=10)
doc_seconds["BM25"] = time.perf_counter() - start

# 평가 질문마다 BM25 상위 10개의 원문 ID를 모으며 질문 1건 평균 시간(ms) 측정
bm25_top_ids = []
start = time.perf_counter()
for question in eval_texts:
    bm25_top_ids.append([doc.metadata["source_id"] for doc in bm25.invoke(question)])
query_ms["BM25"] = round((time.perf_counter() - start) * 1000 / len(eval_texts), 1)
top_ids_by_model["BM25"] = bm25_top_ids
print("BM25 색인 문서:", len(spot_documents), "| 색인 시간(초):", round(doc_seconds["BM25"], 3), "| 질문 1건(ms):", query_ms["BM25"])

**OpenAI 모델.** 문서 388건은 한 번에 보내 시간을 측정하고, 질문은 하나씩 보내며 질문 1건 평균 시간을 측정합니다(네트워크 왕복 포함). 3-large는 과정 기본 768차원과 최대 3072차원을 각각 측정합니다.

In [ ]:
# 3-small 문서 임베딩과 걸린 시간 (임베딩 뒤 정규화)
start = time.perf_counter()
small_doc_vectors = normalize(small_embedding_model.embed_documents(spot_texts))
doc_seconds["3-small (1536)"] = time.perf_counter() - start

# 질문을 하나씩 임베딩하며 질문 1건 평균 시간(ms) 측정, 그다음 상위 10개 문서 찾기
small_question_vectors = []
start = time.perf_counter()
for question in eval_texts:
    small_question_vectors.append(small_embedding_model.embed_query(question))
query_ms["3-small (1536)"] = round((time.perf_counter() - start) * 1000 / len(eval_texts), 1)
small_question_vectors = normalize(small_question_vectors)
top_ids_by_model["3-small (1536)"] = search_top_ids(small_question_vectors, small_doc_vectors, spot_ids)
print("3-small 색인 문서:", len(spot_texts), "| 색인 시간(초):", round(doc_seconds["3-small (1536)"], 3), "| 질문 1건(ms):", query_ms["3-small (1536)"])

In [ ]:
# 3-large 문서 임베딩과 걸린 시간 (과정 기본 설정 768차원)
start = time.perf_counter()
large_doc_vectors = normalize(embedding_model.embed_documents(spot_texts))
doc_seconds["3-large (768)"] = time.perf_counter() - start

# 질문을 하나씩 임베딩하며 질문 1건 평균 시간(ms) 측정, 그다음 상위 10개 문서 찾기
large_question_vectors = []
start = time.perf_counter()
for question in eval_texts:
    large_question_vectors.append(embedding_model.embed_query(question))
query_ms["3-large (768)"] = round((time.perf_counter() - start) * 1000 / len(eval_texts), 1)
large_question_vectors = normalize(large_question_vectors)
top_ids_by_model["3-large (768)"] = search_top_ids(large_question_vectors, large_doc_vectors, spot_ids)
print("3-large(768) 색인 문서:", len(spot_texts), "| 색인 시간(초):", round(doc_seconds["3-large (768)"], 3), "| 질문 1건(ms):", query_ms["3-large (768)"])

In [ ]:
large_3072_model = OpenAIEmbeddings(model="text-embedding-3-large", dimensions=3072, check_embedding_ctx_length=False)  # 같은 3-large를 최대 차원으로

# 3-large 3072차원 문서 임베딩과 걸린 시간
start = time.perf_counter()
large_3072_doc_vectors = normalize(large_3072_model.embed_documents(spot_texts))
doc_seconds["3-large (3072)"] = time.perf_counter() - start

# 질문을 하나씩 임베딩하며 질문 1건 평균 시간(ms) 측정, 그다음 상위 10개 문서 찾기
large_3072_question_vectors = []
start = time.perf_counter()
for question in eval_texts:
    large_3072_question_vectors.append(large_3072_model.embed_query(question))
query_ms["3-large (3072)"] = round((time.perf_counter() - start) * 1000 / len(eval_texts), 1)
large_3072_question_vectors = normalize(large_3072_question_vectors)
top_ids_by_model["3-large (3072)"] = search_top_ids(large_3072_question_vectors, large_3072_doc_vectors, spot_ids)
print("3-large(3072) 색인 문서:", len(spot_texts), "| 색인 시간(초):", round(doc_seconds["3-large (3072)"], 3), "| 질문 1건(ms):", query_ms["3-large (3072)"])

**multilingual-e5-small.** 모델 카드 규약대로 질문에 `query: `, 문서에 `passage: `를 붙입니다.

In [ ]:
from sentence_transformers import SentenceTransformer

me5_model = SentenceTransformer("intfloat/multilingual-e5-small")  # 맨 위에서 받은 캐시를 읽음(없으면 약 0.47GB 내려받음)

# 문서 임베딩과 걸린 시간 (e5는 문서 앞에 "passage: "를 붙임)
start = time.perf_counter()
me5_doc_vectors = me5_model.encode(["passage: " + text for text in spot_texts], normalize_embeddings=True, show_progress_bar=False)
doc_seconds["me5-small (384)"] = time.perf_counter() - start

# 질문을 하나씩 임베딩("query: ")하며 질문 1건 평균 시간(ms) 측정, 그다음 상위 10개 문서 찾기
me5_question_vectors = []
start = time.perf_counter()
for question in eval_texts:
    me5_question_vectors.append(me5_model.encode("query: " + question, normalize_embeddings=True, show_progress_bar=False))
query_ms["me5-small (384)"] = round((time.perf_counter() - start) * 1000 / len(eval_texts), 1)
top_ids_by_model["me5-small (384)"] = search_top_ids(me5_question_vectors, me5_doc_vectors, spot_ids)
print("me5-small 색인 문서:", len(spot_texts), "| 색인 시간(초):", round(doc_seconds["me5-small (384)"], 3), "| 질문 1건(ms):", query_ms["me5-small (384)"])

**KURE-v1.** 접두어 없이 그대로 넣습니다.

In [ ]:
kure_model = SentenceTransformer("nlpai-lab/KURE-v1")  # 맨 위에서 받은 캐시를 읽음(없으면 약 2.3GB 내려받음)

# 문서 임베딩과 걸린 시간 (접두어 없음)
start = time.perf_counter()
kure_doc_vectors = kure_model.encode(spot_texts, normalize_embeddings=True, show_progress_bar=False)
doc_seconds["KURE-v1 (1024)"] = time.perf_counter() - start

# 질문을 하나씩 임베딩하며 질문 1건 평균 시간(ms) 측정, 그다음 상위 10개 문서 찾기
kure_question_vectors = []
start = time.perf_counter()
for question in eval_texts:
    kure_question_vectors.append(kure_model.encode(question, normalize_embeddings=True, show_progress_bar=False))
query_ms["KURE-v1 (1024)"] = round((time.perf_counter() - start) * 1000 / len(eval_texts), 1)
top_ids_by_model["KURE-v1 (1024)"] = search_top_ids(kure_question_vectors, kure_doc_vectors, spot_ids)
print("KURE-v1 색인 문서:", len(spot_texts), "| 색인 시간(초):", round(doc_seconds["KURE-v1 (1024)"], 3), "| 질문 1건(ms):", query_ms["KURE-v1 (1024)"])

### 4-3. 비교표 만들기

선정 기준 세 단계를 적용하려면 후보마다 정확도·시간·비용·잘림 비율이 한 행에 모여 있어야 합니다.

- 정확도와 함께 100만 건 색인 비용(로컬 모델과 BM25는 색인 시간)과 잘림 비율을 계산합니다.
- 잘림 비율은 모델마다 자기 토크나이저로 센 토큰 수가 최대 입력을 넘는 문서의 비율입니다.

In [ ]:
import tiktoken

encoding = tiktoken.encoding_for_model("text-embedding-3-small")  # API가 요금을 매기는 토큰 수와 같은 기준

# 후보별 최대 입력: OpenAI는 공식 문서의 8192 토큰, 로컬 모델은 모델에 설정된 값
max_input_tokens = {"3-small (1536)": 8192, "3-large (768)": 8192, "3-large (3072)": 8192,
                    "me5-small (384)": me5_model.max_seq_length, "KURE-v1 (1024)": kure_model.max_seq_length}

price_per_million_tokens = {"3-small (1536)": 0.02, "3-large (768)": 0.13, "3-large (3072)": 0.13}  # 100만 토큰당 달러, 로컬 모델과 BM25는 없음


def count_tokens(texts):
    """후보 모델별로 문서마다의 토큰 수 목록을 담은 dict를 반환합니다."""
    openai_counts = [len(encoding.encode(text)) for text in texts]
    # 로컬 모델은 자기 토크나이저로 실제 입력(접두어 포함)을 자르지 않고 셈 (verbose=False는 길이 경고 끄기)
    me5_counts = [len(me5_model.tokenizer("passage: " + text, verbose=False)["input_ids"]) for text in texts]
    kure_counts = [len(kure_model.tokenizer(text, verbose=False)["input_ids"]) for text in texts]

    return {"3-small (1536)": openai_counts, "3-large (768)": openai_counts, "3-large (3072)": openai_counts,
            "me5-small (384)": me5_counts, "KURE-v1 (1024)": kure_counts}  # OpenAI 모델은 모두 같은 토크나이저


def truncated_ratio(name, token_counts):
    """후보 name에서 최대 입력보다 길어 뒷부분이 잘리는 문서의 비율을 반환합니다."""
    if name not in max_input_tokens:  # BM25는 최대 입력이 없어 잘림 없음
        return 0.0

    # 최대 입력보다 긴 문서 수의 비율
    over = 0
    for count in token_counts[name]:
        if count > max_input_tokens[name]:
            over += 1
    return round(over / len(token_counts[name]), 3)


token_counts = count_tokens(spot_texts)  # 관광지 문서의 후보별 토큰 수

# 길이 기준마다 평균·최대 길이를 한 행으로 만든 길이 표 (토크나이저마다 토큰 수가 달라 모델별로)
length_rows = []
for name, lengths, limit in [
    ("글자 수(제목 + 본문)", [len(text) for text in spot_texts], "-"),
    ("OpenAI 토큰(tiktoken)", token_counts["3-small (1536)"], 8192),
    ("me5-small 토큰(접두어 포함)", token_counts["me5-small (384)"], max_input_tokens["me5-small (384)"]),
    ("KURE-v1 토큰", token_counts["KURE-v1 (1024)"], max_input_tokens["KURE-v1 (1024)"]),
]:
    length_rows.append({"기준": name, "평균": round(float(np.mean(lengths)), 1), "최대": int(np.max(lengths)), "최대 입력": limit})
display(pd.DataFrame(length_rows))

**길이 표 읽기**
- 같은 문서도 토크나이저마다 토큰 수가 다릅니다. OpenAI 토큰이 로컬 모델의 약 1.7배입니다.
- `최대`가 `최대 입력`을 넘는 것은 me5-small(919 > 512)뿐이라, 이 모델만 긴 문서가 잘립니다.

In [ ]:
# 후보마다 정확도·시간·비용·잘림 비율을 한 행으로 쌓은 비교표
compare_rows = []
for name in ["BM25", "3-small (1536)", "3-large (768)", "3-large (3072)", "me5-small (384)", "KURE-v1 (1024)"]:
    row = evaluate_model(name, top_ids_by_model[name])
    row["query_ms"] = query_ms[name]

    # API 모델은 100만 건 색인(문서 임베딩) 요금, 로컬 모델과 BM25는 요금 대신 100만 건 색인 시간(시간)
    row["cost_per_million_docs_usd"] = 0.0
    row["로컬 100만 건 색인(h)"] = None
    if name in price_per_million_tokens:
        row["cost_per_million_docs_usd"] = round(float(np.mean(token_counts[name])) * price_per_million_tokens[name], 3)
    else:
        row["로컬 100만 건 색인(h)"] = round(doc_seconds[name] / len(spot_texts) * 1_000_000 / 3600, 3)

    # 잘림 비율을 붙여 비교표에 추가
    row["truncated_ratio"] = truncated_ratio(name, token_counts)
    compare_rows.append(row)
display(pd.DataFrame(compare_rows))

`NaN`은 해당 없는 칸(API 모델의 로컬 색인 시간)입니다.

**정확도**
- BM25 20, KURE-v1 21, me5-small 22문항으로 1~2문항 차이라 두 로컬 모델이 BM25보다 낫다고 단정하지 않습니다.
- 과정 기본 `3-large (768)`은 13문항으로 가장 적습니다. 기본값을 믿지 않고 평가셋으로 정확도를 직접 측정하는 이유입니다.

**지연·잘림**
- API 모델의 `query_ms`는 네트워크 왕복이 포함돼 로컬 모델보다 깁니다(로컬 모델은 맥 GPU 기준).
- me5-small의 `truncated_ratio` 0.046은 문서 약 5%의 뒷부분이 잘렸다는 뜻입니다.

In [ ]:
def select_model(rows):
    """선정 기준 세 단계를 차례로 적용해 고른 후보의 행(dict)을 반환합니다. 단계마다 남은 후보를 출력합니다."""
    # BM25는 기준선이라 고를 대상에서 제외
    embedding_rows = []
    for row in rows:
        if row["model"] != "BM25":
            embedding_rows.append(row)

    # 1단계 정확도: 맞힌 문항이 가장 많은 후보에서 2문항 이내면 같은 수준
    best_found = 0
    for row in embedding_rows:
        if row["맞힌 문항"] > best_found:
            best_found = row["맞힌 문항"]
    step1 = []
    for row in embedding_rows:
        if best_found - row["맞힌 문항"] <= 2:
            step1.append(row)
    print("1단계 정확도 통과:", [row["model"] for row in step1])

    # 2단계 입력 길이: 잘림 비율이 1%를 넘는 후보는 제외
    step2 = []
    for row in step1:
        if row["truncated_ratio"] <= 0.01:
            step2.append(row)
    print("2단계 잘림 통과:", [row["model"] for row in step2])

    # 3단계 운영: 로컬 모델(단가표에 없는 모델)을 먼저, 그다음 질문 1건 평균 시간이 짧은 모델
    choice = step2[0]
    for row in step2[1:]:
        row_is_api = row["model"] in price_per_million_tokens
        choice_is_api = choice["model"] in price_per_million_tokens
        if choice_is_api and not row_is_api:
            choice = row
        elif choice_is_api == row_is_api and row["query_ms"] < choice["query_ms"]:
            choice = row
    return choice


travel_choice = select_model(compare_rows)  # 여행지 비교표에 선정 기준 적용
print("선택:", travel_choice["model"])

선정 기준 세 단계를 적용하면 KURE-v1이 남습니다. OpenAI 모델은 가장 많이 맞힌 me5-small보다 2문항 넘게 적고, me5-small은 잘림 비율이 1%를 넘기 때문입니다. 5절에서 차원과 저장 공간을 확인한 뒤 5-3에서 ADR-001로 남깁니다.

## 5. 차원과 저장 공간

4절은 정확도·잘림·질문 시간으로 모델을 골랐습니다. 저장 공간도 네 축의 하나이고 차원에 비례해 커지므로, 차원을 줄여도 정확도가 유지되는지 확인합니다.

### 5-1. 저장 공간 계산

5-2에서 줄일 대상이 얼마나 큰지 알려고, 후보 차원마다 벡터만의 크기를 먼저 계산합니다.

- 표의 차원은 후보 모델들의 차원이고, 값은 벡터만 센 float32 크기입니다(1GB = 10억 바이트). `100만 건`·`1억 건`은 벡터(청크) 수입니다.

In [ ]:
# 후보 모델의 차원마다 float32 벡터 크기(벡터 수 x 차원 x 4바이트) 표 (GB는 10억 바이트)
capacity_rows = []
for dim in [3072, 1536, 1024, 768, 384]:
    capacity_rows.append({
        "차원": dim,
        "벡터 1개(바이트)": dim * 4,
        "100만 건(GB)": round(1_000_000 * dim * 4 / 1e9, 3),
        "1억 건(GB)": round(100_000_000 * dim * 4 / 1e9, 3),
    })
display(pd.DataFrame(capacity_rows))

### 5-2. 같은 모델의 차원 줄이기

과정 기본 768차원이 같은 3-large의 3072차원보다 정확도를 잃는지 확인해야, 줄여서 아낀 저장 공간이 값어치가 있는지 알 수 있습니다. 두 차원의 맞힌 문항은 4절에서 같은 평가셋으로 측정했으므로, 여기서는 그 결과를 저장 공간과 나란히 놓습니다.

**dimensions 인자란?**
- OpenAI 3세대 임베딩(text-embedding-3)은 요청할 때 `dimensions`로 받을 차원을 정할 수 있습니다. Neo4j 벡터 인덱스 단원에서 768로 받은 설정입니다.
- 768차원은 3072차원의 4분의 1이라 벡터 저장 공간도 4분의 1입니다.
- 로컬 후보(KURE-v1 1024, me5-small 384)는 모델마다 차원이 하나로 정해져 있어, 다른 차원이 필요하면 모델을 바꿔 비교합니다.

In [ ]:
# 차원마다 100만 건 저장 공간과 맞힌 문항을 한 행으로 (두 차원 모두 4절의 검색 결과를 그대로 씀)
dimension_rows = []
for name, dim in [("3-large (3072)", 3072), ("3-large (768)", 768)]:
    result = evaluate_model(name, top_ids_by_model[name])
    dimension_rows.append({
        "차원": dim,
        "100만 건(GB)": round(1_000_000 * dim * 4 / 1e9, 3),
        "맞힌 문항": result["맞힌 문항"],
    })
display(pd.DataFrame(dimension_rows))

**표 읽기**
- 768로 줄이면 3문항을 잃어 4절 기준(2문항 이내는 같은 수준)을 넘습니다. 저장 공간을 4분의 1로 줄이는 대신 정확도를 내준 설정입니다.
- 3072차원도 KURE-v1(21문항)보다 적게 맞혀 4절의 선택은 그대로입니다.

### 5-3. 결정 기록: ADR-001 임베딩 모델로 KURE-v1 사용

#### ADR-001: 임베딩 모델로 KURE-v1 사용

- **상태**: 채택
- **맥락**: 여행지 안내 챗봇이 쓸 임베딩 모델을 정해야 하며, 지금은 과정 기본값인 3-large(768)를 쓴다. 질문과 문서를 임베딩할 때 회사 밖 API로 보내지 않고 관광지 개요를 끝까지 읽어야 한다(잘림 1% 이하). 후보는 공개 지표로 좁혔다. MTEB(kor, v2) 검색 점수(Retr.) 1~3위는 문서당 벡터 하나가 아니거나(KURE-v2) 비상업 라이선스이거나(comsat) 4,096차원(Qwen3-Embedding-8B)이라 빼고, 노트북에서 돌고 상업 사용이 되는 KURE-v1(75.90, 10위)을 넣었다. 한국어 점수가 없는 3-small·3-large(768·3072)와 가장 가벼운 me5-small을 더하고 BM25를 기준선으로 둔다.
- **결정**: 여행지 검색의 임베딩 모델로 KURE-v1을 쓴다.
    - 근거: 측정값(관광지 평가셋 30문항, 정답이 상위 5개 안에 들면 맞힌 문항):

        | 후보 | 맞힌 문항 | Recall@5 | 잘림 비율 | 질문 1건(ms) | 외부 API | 판정 |
        |---|---|---|---|---|---|---|
        | BM25 | 20 | 0.667 | 0.000 | 0.5 | 아니오 | 기준선 |
        | 3-small (1536) | 13 | 0.433 | 0.000 | 133.4 | 예 | 정확도 부족 |
        | 3-large (768), 지금 사용 | 13 | 0.433 | 0.000 | 137.1 | 예 | 정확도 부족 |
        | 3-large (3072) | 16 | 0.533 | 0.000 | 148.8 | 예 | 정확도 부족 |
        | me5-small (384) | 22 | 0.733 | 0.046 | 8.7 | 아니오 | 잘림 1% 초과 |
        | **KURE-v1 (1024)** | **21** | **0.700** | **0.000** | **18.6** | 아니오 | **채택** |

        - 3-small·3-large(768): 13문항으로 가장 많이 맞힌 me5-small(22문항)보다 2문항 넘게 적다.
        - 3-large(3072): 16문항으로 역시 2문항 넘게 적다.
        - me5-small: 가장 많이 맞혔지만 잘림 비율이 0.046(문서 약 5%)이라 개요 끝을 읽지 못한다.
        - KURE-v1: 21문항으로 최다와 같은 수준(1문항 차이)이고 잘림이 0이며 로컬에서 돈다.

- **결과**
    - 얻는 것: 지금 쓰는 3-large(768)보다 8문항 더 맞힌다(13 -> 21문항). 질문과 문서를 임베딩할 때 회사 밖 API로 보내지 않고, 개요를 끝까지 읽는다(잘림 0).
    - 감수하는 것: 문서 임베딩을 로컬 장비에서 돌린다. 의미 검색 하나로는 키워드 검색(BM25)보다 1문항만 더 맞힌다(20 -> 21문항). 이름 질문은 둘 다 Recall@5 1.000이고, 차이는 묘사 질문(0.333 -> 0.400)에서만 난다.
    - 다시 검토할 조건:
        - 개요 앞부분만 읽어도 됨 -> me5-small을 후보에 다시 넣어 비교한다.
        - 장비를 둘 수 없어 API 모델만 쓸 수 있음 -> 다른 API 모델의 정확도를 평가셋으로 다시 측정한다(이번 OpenAI 모델은 정확도에서 빠졌다).
        - 평가셋이 바뀜 -> 후보 여섯의 정확도를 다시 측정한다(1~2문항 차이는 뒤집힐 수 있다).

벡터를 어떤 형식으로 저장할지는 교안_02 4-3에서 저장소를 고를 때 정합니다.

## 6. 🖐️ 함께 따라하기: 사내 노무 상담 RAG의 임베딩 고르기

본실습과 같은 비교를 사내 노무 상담이라는 다른 업무에 적용합니다. 리더보드 순위처럼 여행지 결과도 다른 업무에서 그대로 맞는다는 보장이 없어, 업무마다 그 업무의 평가셋으로 정확도를 다시 측정합니다.

**상황**: 인사팀 상담 봇이 근로기준법·시행령과 유연근무 매뉴얼에서 근거를 찾아 답합니다. 지금은 3-small을 쓰는데, 다른 모델로 바꿀 가치가 있는지 평가셋으로 확인합니다.

**평가 질문**
- `직원 말투`, `조문번호`, `필터`(문서 종류를 지정한 질문), `법+매뉴얼`, `2단계 관계` 다섯 유형이고 정답 문서가 둘 이상인 질문도 있습니다.
- `필터` 질문도 여기서는 필터 없이 검색합니다(교안_02도 조건 질문을 필터 없이 측정합니다).

**선정 기준**
1. 외부 API 금지: 정보보안팀은 직원 질문을 회사 밖으로 보내지 않습니다(필수 조건).
2. 잘림 비율 1% 이하: 법무팀은 조문 끝의 단서·예외가 잘리면 안 된다고 봅니다.
3. 남은 후보(BM25 포함) 가운데 맞힌 문항이 가장 많은 모델을 고르고, 같으면 질문 시간이 짧은 쪽을 고릅니다.

본실습은 정확도를 먼저 보고 외부 API는 같은 수준일 때만 따졌지만, 노무는 보안·법무 조건으로 먼저 거른 뒤 정확도로 고릅니다.

진행 순서:

1. 6-1 자료, BM25 기준선, 지금 쓰는 모델 준비(제공)
2. 6-2 후보 평가 함수(직접 설계)
3. 6-3 나머지 네 후보와 비교표(제공)
4. 6-4 노무 기준으로 고르고 노무 ADR-001 남기기(부분 작성)

**실습 진행 방법**

작성 셀은 이 함께 따라하기 절에만 있습니다.

- **직접 설계:** 지시 블록에 적힌 이름·입력·반환값에 맞춰 코드 전체를 작성합니다.
- **부분 작성:** 셀에 남은 `...`만 주석 힌트에 맞춰 채웁니다.
- `...`가 남은 채로 실행하면 에러가 나거나 빈 결과가 나옵니다. 작성 셀을 완성한 뒤 바로 아래 셀을 실행해 결과를 봅니다.
- 완성 코드는 수업이 끝난 뒤 강사가 공유합니다. 먼저 직접 작성해 봅니다.

### 6-1. 자료, BM25 기준선, 지금 쓰는 모델 준비

다른 모델로 바꿀 가치가 있는지 판단하려면 비교 기준이 될 BM25와 지금 쓰는 3-small 결과부터 있어야 합니다.

- `data/따라하기/`의 조문·매뉴얼 250건과 평가 질문을 씁니다.
- 조문 제목(「근로기준법 제60조(연차 유급휴가)」)에 조문번호가 있으므로 본실습처럼 제목과 본문을 이어 넣습니다.

In [ ]:
# 노무 문서와 평가 질문 읽기
labor_docs = [json.loads(line) for line in (data_dir / "따라하기" / "docs.jsonl").read_text(encoding="utf-8").splitlines()]
labor_questions = read_json("따라하기/questions.json")

# 원문 ID, 임베딩 입력(조문번호가 든 제목 + 줄바꿈 + 본문), 질문 문장
labor_ids = [doc["doc_id"] for doc in labor_docs]
labor_texts = [doc["title"] + "\n" + doc["text"] for doc in labor_docs]
labor_question_texts = [question["question"] for question in labor_questions]

print("노무 문서:", len(labor_docs), "| 평가 질문:", len(labor_questions), "문항 | 1문항 =", round(100 / len(labor_questions), 1), "%p")  # 1문항이 움직이는 Recall@5(%p)

In [ ]:
display(pd.DataFrame(labor_questions)["question_type"].value_counts().rename("질문 수").to_frame())  # 유형별 질문 수

In [ ]:
# 후보 이름을 키로 결과 모으기 (뒤의 여섯 후보 비교표가 읽음)
labor_top_ids = {}       # 질문별 상위 10개 원문 ID
labor_doc_seconds = {}   # 문서 전체 색인(문서 임베딩, BM25는 색인 만들기)에 걸린 초
labor_query_ms = {}      # 질문 1건 평균 시간(ms)

# BM25 기준선 색인과 걸린 시간 (본실습과 같은 kiwi_tokenize)
labor_documents = [Document(id=doc_id, page_content=text, metadata={"source_id": doc_id}) for doc_id, text in zip(labor_ids, labor_texts)]
start = time.perf_counter()
labor_bm25 = BM25Retriever.from_documents(labor_documents, preprocess_func=kiwi_tokenize, k=10)
labor_doc_seconds["BM25"] = time.perf_counter() - start

# 평가 질문마다 BM25 상위 10개의 원문 ID를 모으며 질문 1건 평균 시간(ms) 측정
labor_bm25_top_ids = []
start = time.perf_counter()
for question in labor_question_texts:
    labor_bm25_top_ids.append([doc.metadata["source_id"] for doc in labor_bm25.invoke(question)])
labor_query_ms["BM25"] = round((time.perf_counter() - start) * 1000 / len(labor_question_texts), 1)
labor_top_ids["BM25"] = labor_bm25_top_ids
print("BM25 색인 문서:", len(labor_texts), "| 색인 시간(초):", round(labor_doc_seconds["BM25"], 3), "| 질문 1건(ms):", labor_query_ms["BM25"])

In [ ]:
# 상담 봇이 지금 쓰는 3-small: 문서 임베딩과 걸린 시간
start = time.perf_counter()
labor_small_docs = normalize(small_embedding_model.embed_documents(labor_texts))
labor_doc_seconds["3-small (1536)"] = time.perf_counter() - start

# 질문을 하나씩 임베딩하며 질문 1건 평균 시간(ms) 측정, 그다음 상위 10개 문서 찾기
labor_small_questions = []
start = time.perf_counter()
for question in labor_question_texts:
    labor_small_questions.append(small_embedding_model.embed_query(question))
labor_query_ms["3-small (1536)"] = round((time.perf_counter() - start) * 1000 / len(labor_question_texts), 1)
labor_small_questions = normalize(labor_small_questions)
labor_top_ids["3-small (1536)"] = search_top_ids(labor_small_questions, labor_small_docs, labor_ids)
print("3-small 색인 문서:", len(labor_texts), "| 색인 시간(초):", round(labor_doc_seconds["3-small (1536)"], 3), "| 질문 1건(ms):", labor_query_ms["3-small (1536)"])

### 6-2. 후보 평가 함수

본실습의 `evaluate_model`은 평가셋과 유형을 코드에 고정했으므로, 질문 목록을 인자로 받고 유형을 질문에서 읽는 함수를 새로 만듭니다.

**정답이 둘인 질문**
- 하나만 찾으면 Recall@5는 0.5이지만 `맞힌 문항`으로는 셉니다. 그래서 `mean_recall_at_5`가 `맞힌 문항` ÷ 문항 수보다 작을 수 있습니다.

**직접 설계 · 후보 평가 함수**

`evaluate_labor_model(name, top_ids, questions)`를 만듭니다.

- 본실습의 `recall_at_k`, `mrr`를 그대로 씁니다.
- `top_ids`는 `questions`와 같은 순서의 질문별 상위 10개 원문 ID 목록(`labor_top_ids`의 값)입니다.

**작성:**

1. 질문마다 `recall_at_k(..., 5)`로 Recall@5를, `mrr`로 MRR을 구해 모읍니다.
2. Recall@5가 0보다 큰 질문(정답 문서가 하나라도 상위 5개 안)을 `맞힌 문항`으로 셉니다.
3. 유형 이름을 코드에 적지 말고 `question_type`으로 묶어 유형별 Recall@5를 모읍니다.
4. `model`, `mean_recall_at_5`, `mrr`, `맞힌 문항`과 유형마다 `유형 이름 + " R@5"` 키를 담은 dict를 반환합니다. 점수는 소수점 셋째 자리까지 반올림합니다.

**반환값:** 아래 확인 셀에 BM25와 3-small 두 행이 나오고, 열은 `model`, `mean_recall_at_5`, `mrr`, `맞힌 문항`과 유형별 R@5 다섯 개(`직원 말투 R@5`, `조문번호 R@5`, `필터 R@5`, `법+매뉴얼 R@5`, `2단계 관계 R@5`)입니다.

In [ ]:
def evaluate_labor_model(name, top_ids, questions):
    """노무 질문별 검색 결과를 정답과 비교해 비교표의 한 행(dict)을 반환합니다."""
    # 1. 질문마다 recall_at_k(..., 5)로 Recall@5를, mrr로 MRR을 구해 모으세요.
    # 2. Recall@5가 0보다 큰 질문을 맞힌 문항으로 세세요.
    # 3. question_type별로 Recall@5를 모으세요.
    # 4. model, mean_recall_at_5, mrr, 맞힌 문항, 유형별 "유형 이름 R@5" 키를 담은 dict를 반환하세요.
    ...

In [ ]:
display(pd.DataFrame([evaluate_labor_model(name, labor_top_ids[name], labor_questions) for name in labor_top_ids]))  # BM25와 3-small이 한 행씩 나오는지 확인

### 6-3. 나머지 네 후보와 비교표

지금 쓰는 3-small을 바꿀 가치가 있는지 보려면 나머지 네 후보(3-large 768·3072, me5-small, KURE-v1)도 같은 조건으로 정확도와 시간을 측정해야 합니다. 3-large는 본실습처럼 768차원과 3072차원을 따로 측정합니다.

- 본실습과 같은 방법으로 측정하고 여섯 후보 비교표를 만듭니다(제공 코드).
- 질문이 외부 API로 나가는지(`external_api`)도 적습니다.

In [ ]:
# 3-large 문서 임베딩과 걸린 시간 (과정 기본 설정 768차원)
start = time.perf_counter()
labor_large_docs = normalize(embedding_model.embed_documents(labor_texts))
labor_doc_seconds["3-large (768)"] = time.perf_counter() - start

# 질문을 하나씩 임베딩하며 질문 1건 평균 시간(ms) 측정, 그다음 상위 10개 문서 찾기
labor_large_questions = []
start = time.perf_counter()
for question in labor_question_texts:
    labor_large_questions.append(embedding_model.embed_query(question))
labor_query_ms["3-large (768)"] = round((time.perf_counter() - start) * 1000 / len(labor_question_texts), 1)
labor_large_questions = normalize(labor_large_questions)
labor_top_ids["3-large (768)"] = search_top_ids(labor_large_questions, labor_large_docs, labor_ids)
print("3-large(768) 색인 문서:", len(labor_texts), "| 색인 시간(초):", round(labor_doc_seconds["3-large (768)"], 3), "| 질문 1건(ms):", labor_query_ms["3-large (768)"])

In [ ]:
# 3-large 3072차원 문서 임베딩과 걸린 시간 (본실습의 large_3072_model)
start = time.perf_counter()
labor_large_3072_docs = normalize(large_3072_model.embed_documents(labor_texts))
labor_doc_seconds["3-large (3072)"] = time.perf_counter() - start

# 질문을 하나씩 임베딩하며 질문 1건 평균 시간(ms) 측정, 그다음 상위 10개 문서 찾기
labor_large_3072_questions = []
start = time.perf_counter()
for question in labor_question_texts:
    labor_large_3072_questions.append(large_3072_model.embed_query(question))
labor_query_ms["3-large (3072)"] = round((time.perf_counter() - start) * 1000 / len(labor_question_texts), 1)
labor_large_3072_questions = normalize(labor_large_3072_questions)
labor_top_ids["3-large (3072)"] = search_top_ids(labor_large_3072_questions, labor_large_3072_docs, labor_ids)
print("3-large(3072) 색인 문서:", len(labor_texts), "| 색인 시간(초):", round(labor_doc_seconds["3-large (3072)"], 3), "| 질문 1건(ms):", labor_query_ms["3-large (3072)"])

In [ ]:
# me5-small 문서 임베딩("passage: ")과 걸린 시간
start = time.perf_counter()
labor_me5_docs = me5_model.encode(["passage: " + text for text in labor_texts], normalize_embeddings=True, show_progress_bar=False)
labor_doc_seconds["me5-small (384)"] = time.perf_counter() - start

# 질문을 하나씩 임베딩("query: ")하며 질문 1건 평균 시간(ms) 측정, 그다음 상위 10개 문서 찾기
labor_me5_questions = []
start = time.perf_counter()
for question in labor_question_texts:
    labor_me5_questions.append(me5_model.encode("query: " + question, normalize_embeddings=True, show_progress_bar=False))
labor_query_ms["me5-small (384)"] = round((time.perf_counter() - start) * 1000 / len(labor_question_texts), 1)
labor_top_ids["me5-small (384)"] = search_top_ids(labor_me5_questions, labor_me5_docs, labor_ids)
print("me5-small 색인 문서:", len(labor_texts), "| 색인 시간(초):", round(labor_doc_seconds["me5-small (384)"], 3), "| 질문 1건(ms):", labor_query_ms["me5-small (384)"])

In [ ]:
# KURE-v1 문서 임베딩과 걸린 시간 (접두어 없음)
start = time.perf_counter()
labor_kure_docs = kure_model.encode(labor_texts, normalize_embeddings=True, show_progress_bar=False)
labor_doc_seconds["KURE-v1 (1024)"] = time.perf_counter() - start

# 질문을 하나씩 임베딩하며 질문 1건 평균 시간(ms) 측정, 그다음 상위 10개 문서 찾기
labor_kure_questions = []
start = time.perf_counter()
for question in labor_question_texts:
    labor_kure_questions.append(kure_model.encode(question, normalize_embeddings=True, show_progress_bar=False))
labor_query_ms["KURE-v1 (1024)"] = round((time.perf_counter() - start) * 1000 / len(labor_question_texts), 1)
labor_top_ids["KURE-v1 (1024)"] = search_top_ids(labor_kure_questions, labor_kure_docs, labor_ids)
print("KURE-v1 색인 문서:", len(labor_texts), "| 색인 시간(초):", round(labor_doc_seconds["KURE-v1 (1024)"], 3), "| 질문 1건(ms):", labor_query_ms["KURE-v1 (1024)"])

In [ ]:
labor_dimensions = {"BM25": 0, "3-small (1536)": 1536, "3-large (768)": 768, "3-large (3072)": 3072, "me5-small (384)": 384, "KURE-v1 (1024)": 1024}  # 후보별 차원 (BM25는 벡터가 없어 0)

labor_token_counts = count_tokens(labor_texts)  # 잘림 비율을 셀 후보별 문서 토큰 수

# 후보마다 평가 결과와 차원·시간·잘림·외부 API 여부를 한 행으로 쌓은 비교표 (labor_top_ids에 넣은 순서가 행 순서)
labor_compare_rows = []
for name in labor_top_ids:
    row = evaluate_labor_model(name, labor_top_ids[name], labor_questions)
    row["dimension"] = labor_dimensions[name]
    row["doc_embed_seconds"] = round(labor_doc_seconds[name], 3)
    row["query_ms"] = labor_query_ms[name]
    row["truncated_ratio"] = truncated_ratio(name, labor_token_counts)
    row["external_api"] = name in price_per_million_tokens  # 단가표에 있는 OpenAI 모델만 질문을 외부 API로 보냄
    labor_compare_rows.append(row)
display(pd.DataFrame(labor_compare_rows))

- 여섯 후보의 순서를 여행지 비교표와 견주고, 어느 유형의 `R@5` 열에서 후보가 갈리는지 봅니다.
- `맞힌 문항`과 `mean_recall_at_5`의 순서가 같은지도 봅니다.

### 6-4. 노무 기준으로 고르고 노무 ADR-001 남기기

노무는 보안·법무 조건이 먼저라 본실습 `select_model`을 그대로 쓸 수 없으므로, 6절 머리의 선정 기준 세 단계를 함수로 만들어 적용하고, 결과를 노무 ADR-001로 남깁니다.

**부분 작성 · 노무 선정 함수**

`select_labor_model(rows)`의 `...`를 채웁니다. 단계 순서와 출력은 제공했고, 걸러 내는 모양은 본실습 `select_model`과 같습니다.

1. 1단계: `external_api`가 False인 후보만 남깁니다.
2. 2단계: `truncated_ratio`가 0.01 이하인 후보만 남깁니다.
3. 3단계: `맞힌 문항`이 더 많은 후보로 바꾸고, 같으면 `query_ms`가 짧은 후보로 바꿉니다.

**반환값:** 아래 셀에 `1단계 외부 API 통과:`, `2단계 잘림 통과:` 줄로 단계마다 남은 후보가, `결정:` 줄에 고른 모델이 나옵니다.

In [ ]:
def select_labor_model(rows):
    """노무 선정 기준 세 단계를 차례로 적용해 고른 후보의 행(dict)을 반환합니다. 단계마다 남은 후보를 출력합니다."""
    # 1단계 정보보안팀: 외부 API로 질문을 보내지 않는 후보만 남기기
    step1 = []
    for row in rows:
        # external_api 열로 판단하세요.
        if not ...:
            step1.append(row)
    print("1단계 외부 API 통과:", [row["model"] for row in step1])

    # 2단계 법무팀: 잘림 비율 기준을 지키는 후보만 남기기
    step2 = []
    for row in step1:
        # truncated_ratio 열과 선정 기준의 비율로 판단하세요.
        if row["truncated_ratio"] <= ...:
            step2.append(row)
    print("2단계 잘림 통과:", [row["model"] for row in step2])

    # 3단계 정확도: 맞힌 문항이 가장 많은 후보, 같으면 질문 1건 평균 시간이 짧은 후보
    choice = step2[0]
    for row in step2[1:]:
        # 맞힌 문항이 더 많은 후보로 바꾸세요.
        if row["맞힌 문항"] > ...:
            choice = row
        # 맞힌 문항이 같으면 query_ms가 짧은 후보로 바꾸세요.
        elif ...:
            choice = row
    return choice

In [ ]:
labor_choice = select_labor_model(labor_compare_rows)  # 노무 선정 기준 세 단계로 고르기
print("결정:", labor_choice["model"])

단계마다 어느 후보가 빠졌는지 보고, 그 이유를 비교표의 `external_api`, `truncated_ratio`, `맞힌 문항` 열에서 찾습니다.

**작성:** 아래 노무 ADR-001을 더블클릭해 괄호 안 안내를 지우고 다섯 칸을 채웁니다.

#### 노무 ADR-001: (고른 기술이 드러나는 제목)

- **상태**: (제안·채택·거절·대체됨 가운데 하나)
- **맥락**: (2~3문장: 정할 문제와 지금 쓰는 모델, 정보보안팀·법무팀 요구, 검토한 후보. 결정은 쓰지 않음)
- **결정**: (고른 모델을 한 문장으로: …를 쓴다)
    - 근거: (후보마다 한 줄: 뺀 이유. 마지막 줄은 고른 모델이 이긴 이유)

측정값: (후보마다 한 행: 맞힌 문항, Recall@5, 잘림 비율, 외부 API, 판정. 비교표 출력에서 옮깁니다)

- **결과**
    - 얻는 것: (지금 쓰는 모델보다 나아지는 점 등 이 결정으로 얻는 것)
    - 감수하는 것: (대신 감수하는 것)
    - 다시 검토할 조건: (바뀌는 가정이나 수치 -> 다시 할 일)

**확인 퀴즈**: 남은 두 후보의 맞힌 문항 차이가 1문항뿐이라면, 노무 기준과 본실습 기준의 결정은 어떻게 다를까요?

<details><summary>정답 보기</summary>

노무 기준 3단계는 1문항이라도 많은 쪽을 고릅니다. 본실습 기준은 2문항 이내를 같은 수준으로 보고 로컬 여부와 질문 시간으로 고릅니다. 1~2문항 차이는 평가셋이 바뀌면 뒤집힐 수 있으므로, 차이가 작으면 요금과 저장(차원, 저장소의 차원 상한)도 함께 봅니다.

</details>

## 7. 핵심 정리

임베딩 모델을 고르는 순서:

1. 리더보드에서 검색·한국어 점수와 우리 문서와 비슷한 데이터셋의 점수로 후보 서너 개를 좁힙니다(라이선스·크기·차원·가격 확인).
2. 우리 평가셋에서 같은 정확 검색·같은 지표로 정확도를 측정하고 BM25 기준선을 함께 둡니다.
3. 정확도: 맞힌 문항이 가장 많은 후보에서 2문항 이내를 같은 수준으로 봅니다.
4. 입력 길이: 잘림 비율이 1%를 넘는 후보를 뺍니다.
5. 운영: 로컬 모델을 먼저, 그다음 질문 1건 평균 시간이 짧은 모델을 고릅니다.
6. 저장 공간과 저장소의 차원 상한을 보고 필요하면 차원을 줄입니다.
7. 결정마다 제목·상태·맥락·결정·결과 다섯 칸의 ADR로 남깁니다(ADR-001 임베딩 모델로 KURE-v1 사용).

다음 교안에서는 질문 유형마다 검색 전략의 정확도를 비교해 고르고, 그 전략이 보내는 질의로 벡터 저장소를 고른 뒤 아키텍처를 종합 선정하며 ADR-002~004를 이어 씁니다.